# LCEU 文件生成 Notebook

## 概述

本 Notebook 用于生成 **LCEU（土地覆盖生态单元，Land Cover Ecological Unit）栅格文件**。

LCEU 是基于 TEOW（Terrestrial Ecoregions of the World）生态区数据创建的分区栅格，将生态区矢量数据转换为栅格格式，用于后续的土地退化评估分析。

## 工作流程

生成 LCEU 文件的完整流程：

1. **数据准备**：读取 UAE 边界和 TEOW 生态区数据
2. **数据裁剪**：将 TEOW 数据裁剪到 UAE 边界范围内
3. **分区层级选择**：选择合适的分区层级（ECO_ID、BIOME 或 BIOME×REALM）
4. **栅格化**：将矢量 LCEU 数据栅格化到参考栅格坐标系，生成 `lecu.tif` 文件

## 核心函数说明

### `rasterize_lceu_to_ref`

**功能**：将 LCEU 矢量数据栅格化到参考栅格坐标系。

该函数将包含 LCEU 分区信息的 GeoDataFrame 栅格化，使其与参考栅格的空间参考系统、变换矩阵和尺寸对齐，并保存为 GeoTIFF 文件。

**主要参数**：
- `teow_gdf`: GeoDataFrame，必须包含 "lceu_id" 列（整数或可转换为整数）
- `ref_raster_path`: 参考栅格文件路径，用于获取目标 CRS、变换矩阵和尺寸
- `nodata_lceu`: 栅格化时的填充值（默认 0）
- `all_touched`: 是否对所有接触到的像素进行栅格化（默认 False）
- `out_path`: 输出文件路径（可选，如果提供则保存为 GeoTIFF）
- `compress`: 输出文件的压缩方式（默认 "lzw"）

**返回**：形状为 (H, W) 的 int32 数组，包含每个像素的 LCEU ID

**处理流程**：
1. 读取参考栅格的空间参考信息（CRS、变换矩阵、尺寸）
2. 将 GeoDataFrame 投影到参考栅格的 CRS（如果不同）
3. 将矢量多边形栅格化为与参考栅格对齐的栅格
4. 如果提供了输出路径，保存为 GeoTIFF 文件（int32 类型，LZW 压缩）

## LCEU 分区层级选择

LCEU 分区层级决定了如何对生态区进行分类。Notebook 支持三种分区层级：

### 方案 A：ECO_ID（最细粒度）

- **特点**：使用生态区 ID（TEOW 包含约 825 个生态区），最细的分类粒度
- **适用场景**：研究区域较大，能够保证每个生态区有足够的像素
- **代码**：
```python
USE = "ECO_ID"
teow_uae["lceu_id"] = teow_uae["ECO_ID"].round().astype("int32")
```

### 方案 B：BIOME（中等粒度）⭐ 推荐

- **特点**：使用生物群系分类（如沙漠、草原、森林等），平衡了细节和稳定性
- **适用场景**：研究区域较小（如 UAE 全境），**推荐用于 UAE 全境**
- **代码**：
```python
USE = "BIOME"
teow_uae["lceu_id"] = teow_uae["BIOME"].round().astype("int32")
```

### 方案 C：BIOME×REALM（最粗粒度）

- **特点**：结合生物群系（BIOME）和生物地理界（REALM），最稳定
- **编码方式**：`realm_id * 100 + biome_id`（例如 203 = realm 2 + biome 3）
- **适用场景**：研究区域很小，像素数量有限，需要最稳定的分区结果
- **代码**：
```python
USE = "BIOME_REALM"
realms = {r:i+1 for i, r in enumerate(sorted(teow_uae["REALM"].dropna().unique()))}
teow_uae["realm_id"] = teow_uae["REALM"].map(realms).astype("int32")
teow_uae["biome_id"] = teow_uae["BIOME"].round().astype("int32")
teow_uae["lceu_id"] = teow_uae["realm_id"] * 100 + teow_uae["biome_id"]
```

### 选择建议

**对于 UAE 这样的小区域**：
1. **优先使用 BIOME**：平衡了细节和稳定性，推荐作为默认选择
2. **如果像素数量仍然不足**：使用 BIOME×REALM
3. **如果区域足够大**：可以考虑使用 ECO_ID 获得更细的分类

## 输入数据要求

### 必需数据

1. **TEOW 生态区数据**（必需）
   - **格式**：Shapefile
   - **必需字段**：`ECO_ID`（生态区 ID）、`BIOME`（生物群系）、`REALM`（生物地理界）
   - **坐标系**：通常为 EPSG:4326（WGS84）
   - **示例路径**：`../data/LCEU/wwf_terr_ecos.shp`

2. **UAE 边界数据**（必需）
   - **格式**：Shapefile
   - **用途**：用于裁剪 TEOW 数据到研究区域
   - **示例路径**：`../abu_dhabi_all/abu_dhabi_all.shp`

3. **参考栅格数据**（必需）
   - **格式**：GeoTIFF
   - **用途**：提供目标坐标系、变换矩阵和栅格尺寸
   - **示例路径**：`../data/NPP_PROXY/annual_auc_toBands_2000_2015.tif`

## 输出文件说明

### 输出文件：`lecu.tif`

**文件位置**：`../data/LCEU/lecu.tif`（或用户指定的路径）

**文件属性**：
- **数据类型**：int32（整数）
- **nodata 值**：0（表示无 LCEU 分区的区域）
- **坐标系**：与参考栅格相同
- **空间分辨率**：与参考栅格相同
- **尺寸**：与参考栅格相同（H × W）
- **压缩方式**：LZW 压缩
- **分块大小**：512 × 512 像素

**文件内容**：
- 每个像素包含一个 LCEU ID（整数）
- LCEU ID 对应所选分区层级的分类值（ECO_ID、BIOME 或 BIOME×REALM）
- 值为 0 的像素表示该位置不在任何 LCEU 分区内（通常为研究区域外）

**用途**：用于后续的土地退化评估分析，作为生态单元分区的空间参考

## 注意事项

1. **坐标系一致性**：
   - 确保所有输入数据具有正确的 CRS 信息
   - 函数会自动将 GeoDataFrame 投影到参考栅格的 CRS
   - 建议在矢量阶段统一坐标系，避免栅格化时的精度损失

2. **数据对齐**：
   - 输出的 LCEU 栅格与参考栅格具有完全相同的空间参考和尺寸
   - 确保参考栅格覆盖研究区域
   - LCEU 栅格可以直接用于后续与参考栅格对齐的分析

3. **分区层级选择**：
   - 根据研究区域大小选择合适的层级
   - 检查每个分区的像素数量，确保足够用于后续分析
   - 对于小区域（如 UAE），推荐使用 BIOME

4. **无效值处理**：
   - nodata 值为 0，表示该像素不在任何 LCEU 分区内
   - 通常出现在研究区域边界外或数据缺失区域
   - 在后续分析中需要正确处理这些无效值

## 常见问题

### Q1: 如何检查生成的 LCEU 文件是否正确？

```python
import rasterio
import numpy as np

with rasterio.open("../data/LCEU/lecu.tif") as src:
    lceu = src.read(1)
    print("LCEU 唯一值:", np.unique(lceu))
    print("无效像素比例:", (lceu == 0).mean())
    print("有效分区数量:", len(np.unique(lceu[lceu != 0])))
    print("坐标系:", src.crs)
    print("尺寸:", src.width, "×", src.height)
```

### Q2: 如何验证 LCEU 栅格与参考栅格对齐？

```python
import rasterio

with rasterio.open("reference.tif") as ref, \
     rasterio.open("lecu.tif") as lceu:
    assert ref.crs == lceu.crs, "坐标系不一致"
    assert ref.width == lceu.width and ref.height == lceu.height, "尺寸不一致"
    assert ref.transform == lceu.transform, "变换矩阵不一致"
    print("✓ LCEU 栅格与参考栅格对齐")
```

In [2]:
#1) 读 UAE 边界 + TEOW，并裁剪 TEOW 到 UAE
from pathlib import Path
import geopandas as gpd

teow_path = Path("../data/LCEU/wwf_terr_ecos.shp")
uae_path  = Path("../abu_dhabi_all/abu_dhabi_all.shp")   # UAE 国家边界

teow = gpd.read_file(teow_path)        # CRS: EPSG:4326（你已确认）
uae  = gpd.read_file(uae_path)

# 统一 CRS（先在矢量阶段统一到同一 CRS，通常用 teow 的 CRS）
if uae.crs != teow.crs:
    uae = uae.to_crs(teow.crs)

# UAE 可能是多个面，合成一个
uae_geom = uae.union_all()

# 裁剪 TEOW 到 UAE
teow_uae = teow[teow.intersects(uae_geom)].copy()
teow_uae["geometry"] = teow_uae.geometry.intersection(uae_geom)
teow_uae = teow_uae[~teow_uae.is_empty]
print("TEOW polygons in UAE:", len(teow_uae))


TEOW polygons in UAE: 7


2) 选 LCEU 层级：建议 UAE 全境优先用 BIOME（或 BIOME×REALM）

原因：UAE 很小，直接用 ECO_ID 可能只落到少数几个生态区还好，但也可能出现边缘碎片像素太少导致 P90 不稳。最稳妥的起点：

方案 A（更细）：ECO_ID（825 ecoregions）

方案 B（更稳）：BIOME 或 BIOME×REALM

In [3]:
import numpy as np

# 选一种
USE = "BIOME"   # 或 "BIOME" 或 "BIOME_REALM"

if USE == "ECO_ID":
    teow_uae["lceu_id"] = teow_uae["ECO_ID"].round().astype("int32")
elif USE == "BIOME":
    teow_uae["lceu_id"] = teow_uae["BIOME"].round().astype("int32")
elif USE == "BIOME_REALM":
    # REALM 是字符串（如 "PA" 等），先编码成整数
    realms = {r:i+1 for i, r in enumerate(sorted(teow_uae["REALM"].dropna().unique()))}
    teow_uae["realm_id"] = teow_uae["REALM"].map(realms).astype("int32")
    teow_uae["biome_id"] = teow_uae["BIOME"].round().astype("int32")
    teow_uae["lceu_id"]  = teow_uae["realm_id"] * 100 + teow_uae["biome_id"]  # 例如 203=realm2+biome3
else:
    raise ValueError(USE)


In [4]:
import rasterio
from rasterio.features import rasterize
import numpy as np

from __future__ import annotations

from pathlib import Path
import numpy as np
import rasterio
from rasterio.features import rasterize


def rasterize_lceu_to_ref(
    teow_gdf,
    ref_raster_path: str | Path,
    nodata_lceu: int = 0,
    all_touched: bool = False,
    out_path: str | Path | None = None,
    compress: str = "lzw",
) -> np.ndarray:
    """将 LCEU（土地覆盖生态单元）矢量数据栅格化到参考栅格坐标系。

    该函数将包含 LCEU 分区信息的 GeoDataFrame 栅格化，使其与参考栅格的空间参考系统、
    变换矩阵和尺寸对齐。如果提供了输出路径，会将结果保存为 GeoTIFF 文件。

    Args:
        teow_gdf: GeoDataFrame，必须包含 "lceu_id" 列（整数或可转换为整数），
            以及有效的 CRS 信息。
        ref_raster_path: 参考栅格文件路径，用于获取目标 CRS、变换矩阵和尺寸。
        nodata_lceu: 栅格化时的填充值，同时作为输出 TIF 的 nodata 元数据。
            默认为 0。
        all_touched: 是否对所有接触到的像素进行栅格化。默认为 False，
            只栅格化中心点被多边形覆盖的像素。
        out_path: 可选，如果提供，将栅格化结果保存为 GeoTIFF（int32 类型）。
            默认为 None，不保存文件。
        compress: 输出文件的压缩方式。默认为 "lzw"。

    Returns:
        np.ndarray: 形状为 (H, W) 的 int32 数组，包含每个像素的 LCEU ID。

    Raises:
        ValueError: 如果 teow_gdf 没有设置 CRS 或缺少 "lceu_id" 列。

    Example:
        >>> teow_gdf = gpd.read_file("wwf_terr_ecos.shp")
        >>> teow_gdf["lceu_id"] = teow_gdf["ECO_ID"].astype("int32")
        >>> lceu_raster = rasterize_lceu_to_ref(
        ...     teow_gdf,
        ...     ref_raster_path="reference.tif",
        ...     out_path="lceu.tif"
        ... )
    """
    ref_raster_path = Path(ref_raster_path)

    with rasterio.open(ref_raster_path) as src:
        out_shape = (src.height, src.width)
        transform = src.transform
        ref_crs = src.crs
        ref_profile = src.profile.copy()

    # 投影到参考栅格 CRS（避免错位）
    if teow_gdf.crs is None:
        raise ValueError("teow_gdf.crs is None，请先给 GeoDataFrame 设置 CRS")
    if teow_gdf.crs != ref_crs:
        teow_gdf = teow_gdf.to_crs(ref_crs)

    if "lceu_id" not in teow_gdf.columns:
        raise ValueError('teow_gdf must contain column "lceu_id"')

    # 确保可转 int
    vals = teow_gdf["lceu_id"].values
    # 如果是 float（如 61404.0），round 后转 int 更稳
    vals_int = np.asarray(np.round(vals), dtype=np.int32)

    shapes = [
        (geom, int(val))
        for geom, val in zip(teow_gdf.geometry, vals_int)
        if geom is not None and (not geom.is_empty)
    ]

    lceu_id = rasterize(
        shapes=shapes,
        out_shape=out_shape,
        transform=transform,
        fill=nodata_lceu,
        dtype="int32",
        all_touched=all_touched,
    )

    # ✅ 直接保存
    if out_path is not None:
        out_path = Path(out_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)

        profile = ref_profile
        profile.update(
            driver="GTiff",
            count=1,
            dtype="int32",
            nodata=nodata_lceu,
            compress=compress,
            tiled=True,
            blockxsize=512,
            blockysize=512,
        )

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(lceu_id.astype(np.int32, copy=False), 1)

    return lceu_id


nodata_lceu = 0
lceu_id = rasterize_lceu_to_ref(
    teow_uae,
    ref_raster_path="../data/NPP_PROXY/annual_auc_toBands_2000_2015.tif",
    nodata_lceu=0,
    out_path="../data/LCEU/lecu.tif",
)

print("lceu_id unique (head):", np.unique(lceu_id)[:20])
print("nodata ratio:", (lceu_id == nodata_lceu).mean())


lceu_id unique (head): [ 0  8 13]
nodata ratio: 0.6309652655122101
